# 02: Running the Agent

In Notebook 01 we explored the dataset and tools. This notebook shows how to run the
`KnowledgeGroundedAgent` in practice.

## What You'll Learn

1. The agent's PlanReAct architecture and the `AgentResponse` data structure
2. Running a question with live progress display
3. Inspecting the response: plan, tool calls, sources, and reasoning
4. Multi-turn conversations using session state
5. Observability with Langfuse tracing

## Prerequisites

Complete Notebook 01. You'll need `GOOGLE_API_KEY` in your `.env` file.
For tracing (Section 4): `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` are also required.

In [1]:
import os
import uuid
from pathlib import Path

from aieng.agent_evals.knowledge_qa import KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from aieng.agent_evals.langfuse import init_tracing
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


# Set working directory to the repository root
if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent)
    print(f"Working directory set to: {Path('').absolute()}")


load_dotenv(verbose=True)
console = Console(width=100)

Working directory set to: /home/coder/eval-agents


## 1. Agent Architecture

The `KnowledgeGroundedAgent` is built on Google ADK and combines two patterns:

**PlanReAct** — Before executing, the agent produces an explicit research plan with numbered
steps. Each step has a type (`SEARCH`, `FETCH`, `ANALYZE`) and a status that transitions from
`pending` → `in_progress` → `completed` (or `failed`/`skipped`). The plan can be revised
mid-run if the agent encounters unexpected results.

**ReAct loop** — Within each step, the agent alternates between reasoning (Thought), acting
(tool call), and observing (tool response).

### The `AgentResponse` Object

After running, `agent.answer_async(question)` returns an `AgentResponse`:

| Field | Type | Description |
|-------|------|-------------|
| `text` | `str` | The final answer |
| `plan` | `ResearchPlan` | Numbered steps with statuses |
| `tool_calls` | `list[dict]` | Every tool invocation during execution |
| `sources` | `list[GroundingChunk]` | URLs used as evidence |
| `reasoning_chain` | `list[str]` | The model's intermediate reasoning |
| `total_duration_ms` | `int` | Wall-clock execution time |

In [2]:
agent = KnowledgeGroundedAgent(enable_planning=True)

config_table = Table(title="Agent Configuration", show_header=False)
config_table.add_column("Setting", style="cyan")
config_table.add_column("Value", style="white")
config_table.add_row("Model", agent.model)
config_table.add_row("Planning", "PlanReAct (enabled)")
config_table.add_row("Session Service", "InMemorySessionService")
config_table.add_row("Tools", "google_search, web_fetch, fetch_file, grep_file, read_file")

console.print(config_table)

                              Agent Configuration                               
┌─────────────────┬────────────────────────────────────────────────────────────┐
│ Model           │ gemini-2.5-flash                                           │
│ Planning        │ PlanReAct (enabled)                                        │
│ Session Service │ InMemorySessionService                                     │
│ Tools           │ google_search, web_fetch, fetch_file, grep_file, read_file │
└─────────────────┴────────────────────────────────────────────────────────────┘

## 2. Running a Question

`run_with_display` executes the agent in a Jupyter notebook with a live progress display showing:

- The research plan with step statuses (updating in real time)
- Tool calls as they fire

We'll use a question that requires web search — the agent must find and verify a specific fact,
not recall it from training data.

In [3]:
question = "Any news regarding earning calls from Canada's Big Five? Summarize if there is any."

response = await run_with_display(agent, question)

In [4]:
display_response(
    console,
    response.text,
    subtitle=(
        f"Duration: {response.total_duration_ms / 1000:.1f}s  |  "
        f"Tool calls: {len(response.tool_calls)}  |  "
        f"Sources: {len(response.sources)}"
    ),
)

╭───────────────────────────────────────────── Answer ─────────────────────────────────────────────╮
│ Here's a summary of the recent earnings calls from Canada's Big Five banks for their first       │
│ quarter of fiscal year 2026 (Q1 2026), with most calls occurring in late February 2026:          │
│                                                                                                  │
│  • Royal Bank of Canada (RBC): Reported record Q1 2026 results with a profit of $5.79 billion, a │
│    13% increase year-over-year, and adjusted diluted EPS of $4.08, surpassing analyst estimates. │
│    Revenue reached $17.96 billion, with strong performances across all key segments.             │
│  • Toronto-Dominion Bank (TD): Announced record Q1 2026 earnings of C$4.2 billion and an EPS of  │
│    C$2.44, exceeding forecasts, with an 11% year-over-year revenue growth. The bank highlighted  │
│    robust trading and fee income growth and margin expansion. Their Q2 2026 conference call is   │
│    scheduled for May 28, 2026.                                                                   │
│  • Bank of Nova Scotia (Scotiabank): Reported a strong start to 2026 with Q1 net income of       │
│    $2,299 million (adjusted $2,695 million) and adjusted diluted EPS of $2.05, surpassing        │
│    expectations. Revenue grew by 11% year-over-year to CAD 10.08 billion. Despite strong         │
│    earnings, the stock dipped due to investor concerns over rising credit loss provisions and    │
│    increased expenses.                                                                           │
│  • Bank of Montreal (BMO): Reported strong Q1 2026 results, with reported net income increasing  │
│    by 16% to $2,489 million and adjusted EPS rising by 15% to $3.48, both surpassing analyst     │
│    expectations. Revenue for the quarter was $9.82 billion, driven by fee growth and margin      │
│    expansion. Their Q2 2026 earnings are expected in late May or early June 2026.                │
│  • Canadian Imperial Bank of Commerce (CIBC): Announced a significant jump in Q1 2026 profit to  │
│    $3.10 billion, with adjusted diluted EPS up 25% to $2.76. The bank achieved record revenue of │
│    $8.40 billion across all business units and an adjusted return on common shareholders' equity │
│    (ROE) of 17.4%.                                                                               │
╰─────────────────────── Duration: 159.7s  |  Tool calls: 6  |  Sources: 36 ───────────────────────╯

╭─────────────────────────────────────────── Reasoning ────────────────────────────────────────────╮
│ The search snippets directly provided comprehensive summaries of the Q1 2026 earnings calls for  │
│ each of Canada's Big Five banks. The information included net income, EPS, revenue, key          │
│ performance indicators, and forward-looking statements where available. This allowed for a       │
│ direct summarization without needing to fetch the full content of the web pages.                 │
│                                                                                                  │
│ NEWS REGARDING EARNINGS CALLS FROM CANADA'S BIG FIVE BANKS                                       │
│                                                                                                  │
│ Here's a summary of the recent earnings calls from Canada's Big Five banks for their first       │
│ quarter of fiscal year 2026 (Q1 2026), with most calls occurring in late February 2026:          │
│                                                                                                  │
│ Royal Bank of Canada (RBC)                                                                       │
│                                                                                                  │
│ RBC reported record Q1 2026 results, with a profit of $5.79 billion, marking a 13% increase      │
│ year-over-year. Their adjusted diluted earnings per share (EPS) of $4.08 surpassed analyst       │
│ estimates, and revenue reached $17.96 billion. The bank saw strong performances across all its   │
│ key segments, including Personal Banking, Commercial Banking, Wealth Management, and Capital     │
│ Markets. The provision for credit losses for the quarter was $1.09 billion.                      │
│                                                                                                  │
│ Toronto-Dominion Bank (TD)                                                                       │
│                                                                                                  │
│ TD held its Q1 2026 earnings call on February 26, 2026, reporting record earnings of C$4.2       │
│ billion and an EPS of C$2.44, both exceeding forecasts. The bank experienced an 11%              │
│ year-over-year revenue growth and achieved a return on equity (ROE) of 14.2%. Highlights         │
│ included robust trading and fee income growth, increased volume in Canadian Personal and         │
│ Commercial Banking, and margin expansion. TD also completed an C$8 billion share buyback and     │
│ initiated a new C$7 billion share buyback program. Their Q2 2026 conference call is scheduled    │
│ for May 28, 2026.                                                                                │
│                                                                                                  │
│ Bank of Nova Scotia (Scotiabank)                                                                 │
│                                                                                                  │
│ Scotiabank announced its Q1 2026 results on February 24, 2026, reporting a net income of $2,299  │
│ million (adjusted $2,695 million) and adjusted diluted EPS of $2.05, both surpassing             │
│ expectations. Revenue grew by 11% year-over-year to CAD 10.08 billion. The adjusted ROE stood at │
│ 13.0%, and the Common Equity Tier 1 (CET1) ratio was 13.3%. Despite strong earnings, the bank's  │
│ stock saw a slight dip, reflecting investor concerns over rising credit loss provisions and      │
│ increased expenses.                                                                              │
│                                                                                                  │
│ Bank of Montreal (BMO)                                                                           │
│                                                            

### 2.1 Inspecting the Response

The `AgentResponse` object contains the full execution trace.

In [5]:
plan = response.plan

plan_table = Table(title="Research Plan")
plan_table.add_column("#", style="cyan", width=3)
plan_table.add_column("Step", style="white")
plan_table.add_column("Type", style="dim", width=12)
plan_table.add_column("Status", style="green")

for step in plan.steps:
    icon = {"completed": "✓", "failed": "✗", "skipped": "○"}.get(step.status.value, "·")
    desc = step.description[:70] + "..." if len(step.description) > 70 else step.description
    plan_table.add_row(str(step.step_id), desc, step.step_type, f"{icon} {step.status.value}")

console.print(plan_table)

                                           Research Plan                                            
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ #   ┃ Step                                                          ┃ Type         ┃ Status      ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 1   │ Search for "Canada's Big Five banks" to confirm the list of   │ research     │ ✓ completed │
│     │ banks.                                                        │              │             │
│ 2   │ Search for "earnings calls" or "financial results" for each   │ research     │ ✓ completed │
│     │ of the ide...                                                 │              │             │
│ 3   │ From the search results, identify relevant URLs that discuss  │ research     │ ○ skipped   │
│     │ recent ea...                                                  │              │             │
│ 4   │ Fetch the content of these relevant URLs using `web_fetch` to │ research     │ ○ skipped   │
│     │ get the ...                                                   │              │             │
│ 5   │ Summarize the key information from the fetched articles       │ research     │ ○ skipped   │
│     │ regarding the ...                                             │              │             │
└─────┴───────────────────────────────────────────────────────────────┴──────────────┴─────────────┘

In [6]:
if response.tool_calls:
    tools_table = Table(title="Tool Calls")
    tools_table.add_column("#", style="dim", width=3)
    tools_table.add_column("Tool", style="cyan", width=16)
    tools_table.add_column("Arguments (truncated)", style="white")

    for i, tc in enumerate(response.tool_calls[:15], 1):
        name = tc.get("name", "unknown")
        args = str(tc.get("args", {}))
        args = args[:70] + "..." if len(args) > 70 else args
        tools_table.add_row(str(i), name, args)

    if len(response.tool_calls) > 15:
        tools_table.add_row("...", f"({len(response.tool_calls) - 15} more)", "")

    console.print(tools_table)
else:
    console.print("[dim]No tool calls recorded[/dim]")

                                             Tool Calls                                             
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Tool             ┃ Arguments (truncated)                                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ google_search    │ {'query': "Canada's Big Five banks"}                                    │
│ 2   │ google_search    │ {'query': 'Royal Bank of Canada (RBC) earnings call news March 2026     │
│     │                  │ OR...                                                                   │
│ 3   │ google_search    │ {'query': 'Toronto-Dominion Bank (TD) earnings call news March 2026     │
│     │                  │ OR...                                                                   │
│ 4   │ google_search    │ {'query': 'Bank of Nova Scotia (Scotiabank) earnings call news March    │
│     │                  │ 2...                                                                    │
│ 5   │ google_search    │ {'query': 'Bank of Montreal (BMO) earnings call news March 2026 OR Q1   │
│     │                  │ ...                                                                     │
│ 6   │ google_search    │ {'query': 'Canadian Imperial Bank of Commerce (CIBC) earnings call      │
│     │                  │ new...                                                                  │
└─────┴──────────────────┴─────────────────────────────────────────────────────────────────────────┘

In [7]:
if response.sources:
    seen: set[str] = set()
    sources_table = Table(title="Sources")
    sources_table.add_column("#", style="dim", width=3)
    sources_table.add_column("URL", style="blue")

    for src in response.sources:
        if src.uri and src.uri not in seen:
            seen.add(src.uri)
            url = src.uri[:85] + "..." if len(src.uri) > 85 else src.uri
            sources_table.add_row(str(len(seen)), url)
        if len(seen) >= 10:
            break

    console.print(sources_table)
else:
    console.print("[dim]No sources recorded[/dim]")

                                             Sources                                              
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ URL                                                                                      ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ https://en.wikipedia.org/wiki/Big_Five_banks_of_Canada                                   │
│ 2   │ https://ca.investing.com/news/transcripts/earnings-call-transcript-td-bank-posts-reco... │
│ 3   │ https://ca.investing.com/news/transcripts/earnings-call-transcript-royal-bank-of-cana... │
│ 4   │ https://www.gurufocus.com/news/8652483/bank-of-montreal-bmo-q1-2026-earnings-call-hig... │
│ 5   │ https://za.investing.com/news/transcripts/earnings-call-transcript-cibc-reports-robus... │
│ 6   │ https://www.investing.com/news/transcripts/earnings-call-transcript-bank-of-montreal-... │
│ 7   │ https://www.marketbeat.com/instant-alerts/bank-of-nova-scotia-q1-earnings-call-highli... │
│ 8   │ https://wiki.c2.com/?BigFiveBanksOfCanada                                                │
│ 9   │ https://public.com/stocks/td/earnings                                                    │
│ 10  │ https://ca.investing.com/news/transcripts/earnings-call-transcript-scotiabank-beats-q... │
└─────┴──────────────────────────────────────────────────────────────────────────────────────────┘

## 3. Multi-Turn Conversations

The agent uses an `InMemorySessionService` to maintain conversation context across turns.
Pass the same `session_id` to link questions together — the agent will use prior context
when answering follow-up questions.

In [9]:
session_id = str(uuid.uuid4())
console.print(f"Session ID: [dim]{session_id}[/dim]\n")

# First turn: establish a subject
response1 = await agent.answer_async(
    "Any news regarding earning calls from Canada's Big Five? Summarize if there is any.",
    session_id=session_id,
)
display_response(console, response1.text, title="Turn 1")

# Second turn: follow-up that references the prior context
response2 = await agent.answer_async(
    "How is each bank's performance compared to that of last quarter or year?",
    session_id=session_id,
)
display_response(console, response2.text, title="Turn 2 (follow-up)")

Session ID: 3a893d48-fdaa-4804-85e4-8eb67c0c974e

2026-03-19 19:11:53,702 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: Any news regarding earning calls from Canada's Big Five? Summarize if there is any....
2026-03-19 19:11:53,840 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-19 19:11:54,970 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-19 19:11:54,974 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 5 steps
2026-03-19 19:11:54,975 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: google_search({'query': "Canada's Big Five banks earning calls recent news"})
2026-03-19 19:11:55,100 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-19 19:12:05,263 INFO aieng.agent_evals.knowledge_qa.agent: Tool response: google_search completed
2026-03-19 19:12:05,800 INFO google_adk.google.adk.models.gemini_context_cache_manager: Cache created suc

╭───────────────────────────────────────────── Turn 1 ─────────────────────────────────────────────╮
│ Canada's major banks, commonly referred to as the "Big Five" (which frequently expands to the    │
│ "Big Six" to include National Bank of Canada), have reported a strong start to fiscal year 2026. │
│ In their first-quarter earnings calls, predominantly held in February 2026, these banks          │
│ collectively surpassed earnings expectations, achieving an approximate combined profit of $19    │
│ billion. This significant increase from around $14 billion in the same period last year was      │
│ largely driven by effective capital management and diversified revenue streams.                  │
│                                                                                                  │
│ Here's a summary of the recent earnings call highlights for some of the Big Five:                │
│                                                                                                  │
│  • Royal Bank of Canada (RBC): RBC's quarterly profit climbed to $5.79 billion, an increase from │
│    $5.13 billion year-over-year. While performing strongly, the bank noted some elevated credit  │
│    losses, particularly a $485 million increase in gross impaired loans from the previous        │
│    quarter, largely from Canadian residential mortgages. RBC increased its provisions for credit │
│    losses to $1.09 billion for the quarter.                                                      │
│  • TD Bank Group (TD): TD also reported increased profits from the previous year across its      │
│    businesses. The bank reduced its provisions for credit losses during the quarter, attributing │
│    this to easing concerns among clients and in the broader economy.                             │
│  • Bank of Montreal (BMO): BMO's profits rose and exceeded expectations, even after accounting   │
│    for staff layoffs. The bank reported a first-quarter profit of $2.49 billion, up from $2.14   │
│    billion in the same period last year.                                                         │
│  • Canadian Imperial Bank of Commerce (CIBC): CIBC's profits were up from a year earlier,        │
│    bolstered by trading revenue and other market-related gains. Its increased exposure to the    │
│    U.S. market, particularly its U.S. capital markets division, saw a 50% increase from the      │
│    prior year, contributing significantly to its earnings. CIBC also reduced its provisions for  │
│    credit losses, with earnings growing 25% year-over-year and a return on equity exceeding 17%. │
│  • Scotiabank: Scotiabank's first-quarter net income surged to $2.3 billion, a substantial rise  │
│    from $993 million a year ago. This increase is partly due to the previous year's results      │
│    being impacted by a $1.4 billion impairment charge. Its Canadian banking division reported a  │
│    profit of $960 million, a five percent increase from the same period last year.               │
│                                                                                                  │
│ Overall, the sentiment from the banks was one of cautious optimism despite ongoing economic      │
│ uncertainties such as elevated unemployment and trade issues. Revenue growth remained strong and │
│ broad-based, encompassing capital markets, net interest income, and fee income. While credit     │
│ deterioration was noted, credit losses were deemed manageable and outweighed by strong revenue   │
│ growth.                                                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-03-19 19:12:23,737 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: How is each bank's performance compared to that of last quarter or year?...
2026-03-19 19:12:23,877 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-19 19:12:23,878 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-19 19:12:33,745 INFO google_adk.google.adk.models.google_llm: Response received from the model.


╭─────────────────────────────────────── Turn 2 (follow-up) ───────────────────────────────────────╮
│ Here's how each of Canada's Big Five banks performed in their first quarter of fiscal year 2026, │
│ compared to the last quarter or year, based on recent earning calls:                             │
│                                                                                                  │
│ Overall Big Six Banks:                                                                           │
│                                                                                                  │
│  • The Big Six banks collectively achieved an approximate combined profit of $19 billion in Q1   │
│    2026, a significant increase from around $14 billion in the same period last year.            │
│                                                                                                  │
│ Royal Bank of Canada (RBC):                                                                      │
│                                                                                                  │
│  • Year-over-year Profit: RBC's quarterly profit climbed to $5.79 billion, an increase from      │
│    $5.13 billion a year earlier. Earnings per share for the period ending January 31, 2026, rose │
│    to $4.03, up from $3.54 in Q1 2025.                                                           │
│  • Quarter-over-quarter Credit Losses: Gross impaired loans increased by $485 million from the   │
│    previous quarter, largely driven by Canadian residential mortgages.                           │
│  • Provisions for Credit Losses: RBC increased its provisions for credit losses to $1.09 billion │
│    for the quarter.                                                                              │
│                                                                                                  │
│ TD Bank Group (TD):                                                                              │
│                                                                                                  │
│  • Year-over-year Profit: TD also reported increased profits from a year earlier, with strong    │
│    results across its businesses. (Specific profit figures for year-over-year comparison were    │
│    not detailed in the summary, but an increase was noted).                                      │
│  • Provisions for Credit Losses: The bank reduced its provisions for credit losses during the    │
│    quarter, citing easing concerns.                                                              │
│                                                                                                  │
│ Bank of Montreal (BMO):                                                                          │
│                                                                                                  │
│  • Year-over-year Profit: BMO's first-quarter profit rose to $2.49 billion, an increase from     │
│    $2.14 billion in the same period last year.                                                   │
│                                                                                                  │
│ Canadian Imperial Bank of Commerce (CIBC):                                                       │
│                                                                                                  │
│  • Year-over-year Profit: CIBC reported profits up from a year earlier, with its earnings in the │
│    most recent quarter growing 25% year-over-year.                                               │
│  • U.S. Capital Markets: Its U.S. capital markets division saw a 50% increase from the prior     │
│    year.                                                                                         │
│  • Provisions for Credit Losses: The bank also reduced its provisions for credit losses.         │
│                                                            

## 4. Observability with Langfuse

Langfuse captures a full trace of every agent run using OpenTelemetry, giving you visibility into:

- Every tool call and its arguments
- Every LLM call with prompts and completions
- Timing for each span
- The full agent execution tree

This is essential for debugging failures, measuring latency, and comparing configurations.

### Trace Structure

```
Trace: agent run
├── Span: planning (PlanReAct)
│   └── LLM Call: create_plan
├── Span: step-1-execution
│   ├── Tool Call: google_search
│   ├── Tool Call: web_fetch
│   └── LLM Call: step_summary
├── Span: step-2-execution
│   └── ...
└── Span: synthesis
    └── LLM Call: final_answer
```

### Prerequisites

Set these in your `.env` file:
- `LANGFUSE_PUBLIC_KEY`
- `LANGFUSE_SECRET_KEY`
- `LANGFUSE_HOST` (optional, defaults to `https://cloud.langfuse.com`)

In [9]:
langfuse_configured = all(
    [
        os.getenv("LANGFUSE_PUBLIC_KEY"),
        os.getenv("LANGFUSE_SECRET_KEY"),
    ]
)

if langfuse_configured:
    console.print("[green]✓[/green] Langfuse credentials found")
else:
    console.print("[yellow]⚠[/yellow] Langfuse credentials not found — tracing cells will be skipped")
    console.print("[dim]Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY in .env[/dim]")

✓ Langfuse credentials found

In [10]:
tracing_enabled = init_tracing()

if tracing_enabled:
    console.print("[green]✓[/green] Langfuse tracing initialized")
else:
    console.print("[yellow]⚠[/yellow] Tracing not enabled (check credentials)")

2026-02-25 19:33:06,140 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-02-25 19:33:06,321 INFO aieng.agent_evals.langfuse: Langfuse tracing initialized successfully (endpoint: https://us.cloud.langfuse.com/api/public/otel)


✓ Langfuse tracing initialized

In [11]:
if tracing_enabled:
    from langfuse import Langfuse

    langfuse = Langfuse()
    traced_agent = KnowledgeGroundedAgent(enable_planning=True)
    traced_question = "What programming language was created by Guido van Rossum, and in what year?"

    console.print(Panel(traced_question, title="Traced Question", border_style="green"))

    with langfuse.start_as_current_span(name="knowledge-agent", input=traced_question):
        trace_id = langfuse.get_current_trace_id()
        traced_response = await traced_agent.answer_async(traced_question)
        langfuse.update_current_span(output=traced_response.text)

    display_response(
        console,
        traced_response.text,
        subtitle=f"Duration: {traced_response.total_duration_ms / 1000:.1f}s",
    )
else:
    console.print("[dim]Skipping (Langfuse not configured)[/dim]")
    trace_id = None

╭──────────────────────────────────────── Traced Question ─────────────────────────────────────────╮
│ What programming language was created by Guido van Rossum, and in what year?                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-02-25 19:33:06,548 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: What programming language was created by Guido van Rossum, and in what year?...


/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/models/google_llm.py:172: UserWarning: [EXPERIMENTAL] GeminiContextCacheManager: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  cache_manager = GeminiContextCacheManager(self.api_client)
2026-02-25 19:33:06,705 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-02-25 19:33:09,469 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-02-25 19:33:09,475 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 4 steps
2026-02-25 19:33:09,477 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: google_search({'query': 'programming language created by Guido van Rossum and in what year'})
2026-02-25 19:33:09,601 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-02-25 19:33:

╭───────────────────────────────────────────── Answer ─────────────────────────────────────────────╮
│ The programming language created by Guido van Rossum is Python. He began working on it in        │
│ December 1989, and it was first released in 1991.                                                │
╰──────────────────────────────────────── Duration: 12.2s ─────────────────────────────────────────╯

╭─────────────────────────────────────────── Reasoning ────────────────────────────────────────────╮
│ The Google search results and the fetched Wikipedia page consistently state that Guido van       │
│ Rossum created Python, with implementation starting in December 1989 and the first release in    │
│ 1991.                                                                                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
if tracing_enabled:
    from IPython.display import HTML, display  # noqa: A004
    from langfuse import Langfuse
    from opentelemetry import trace as otel_trace

    provider = otel_trace.get_tracer_provider()
    if hasattr(provider, "force_flush"):
        provider.force_flush(timeout_millis=5000)
    console.print("[green]✓[/green] Traces flushed to Langfuse")

    if trace_id:
        trace_url = Langfuse().get_trace_url(trace_id=trace_id)
        display(HTML(f'<p>View trace: <a href="{trace_url}" target="_blank">{trace_url}</a></p>'))

✓ Traces flushed to Langfuse

### 4.1 Viewing Traces in the Langfuse UI

Open your Langfuse project and navigate to **Traces**. Each run appears as a
tree of spans. Useful things to look at:

- **Span timeline** — which steps take the most time?
- **Tool call arguments** — what search queries did the agent use?
- **LLM interactions** — what did the model reason about before calling each tool?
- **Errors** — red spans show where failures occurred

You can also filter by trace name, time range, or input content.

## Summary

In this notebook you learned:

1. **Creating the agent** — `KnowledgeGroundedAgent(enable_planning=True)` with PlanReAct
2. **Running questions** — `run_with_display` for live notebook progress; `agent.answer_async` for raw access
3. **The `AgentResponse`** — plan, tool calls, sources, reasoning, and timing in one object
4. **Multi-turn conversations** — linking turns with `session_id`
5. **Langfuse tracing** — `init_tracing()` and the Langfuse SDK for full observability

**Next:** In Notebook 03, we'll run a systematic evaluation using the DeepSearchQA benchmark.

In [13]:
console.print(
    Panel(
        "[green]✓[/green] Notebook complete!\n\n"
        "[cyan]Next:[/cyan] Open [bold]03_evaluation.ipynb[/bold] to evaluate the agent at scale.",
        title="Done",
        border_style="green",
    )
)

╭────────────────────────────────────────────── Done ──────────────────────────────────────────────╮
│ ✓ Notebook complete!                                                                             │
│                                                                                                  │
│ Next: Open 03_evaluation.ipynb to evaluate the agent at scale.                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯